In [1]:
import pandas as pd
import sys

print(sys.version)
print(pd.__version__)

3.14.2 (v3.14.2:df793163d58, Dec  5 2025, 12:18:06) [Clang 16.0.0 (clang-1600.0.26.6)]
3.1.1


In [3]:
rentals_raw = pd.read_csv('data/bike_rentals.csv')
rentals = rentals_raw.copy()

rentals.shape

(403, 15)

In [4]:
rentals.columns[rentals.isna().all()]

Index(['coupon_code', 'damage_notes', 'late_fee_note'], dtype='str')

In [5]:
rentals = rentals.dropna(axis=1, how='all')

rentals.shape

(403, 12)

In [6]:
rentals['price'].head()

0      SEK 855.50
1    SEK 1,311.50
2      SEK 350.00
3      SEK 569.50
4      SEK 828.00
Name: price, dtype: str

In [7]:
rentals['price'] = rentals['price'].str.replace('SEK', '', regex=False).str.strip()
rentals['price'].head()

0      855.50
1    1,311.50
2      350.00
3      569.50
4      828.00
Name: price, dtype: str

In [8]:
rentals['price'] = rentals['price'].str.replace(',', '', regex=False)
rentals['price'].head()

0     855.50
1    1311.50
2     350.00
3     569.50
4     828.00
Name: price, dtype: str

In [9]:
rentals['price'] = rentals['price'].astype(float)
rentals['price'].dtype

dtype('float64')

In [10]:
rentals['price'].describe()

count     352.000000
mean      847.348011
std       633.435513
min       154.500000
25%       406.000000
50%       668.250000
75%      1194.625000
max      3401.000000
Name: price, dtype: float64

In [11]:
rentals.sort_values(['price'], ascending=False).head(3)[['rental_id', 'bike_type', 'status', 'price']]

,rental_id,bike_type,status,price
187,R-5282,Cargo bike,Completed,3401.0
49,R-5209,Cargo bike,Completed,2979.5
270,R-5378,E-bike,Completed,2941.0


In [12]:
rentals.nlargest(3, 'price')

,rental_id,station,bike_type,helmets_text,is_member,insured,paid_online,status,start_date,end_date,price,rating
187,R-5282,Kungsholmen,Cargo bike,No helmets,f,t,t,Completed,2026-08-22,2026-08-29,3401.0,5.0
49,R-5209,Kungsholmen,Cargo bike,No helmets,f,f,t,Completed,2026-03-23,2026-03-30,2979.5,5.0
270,R-5378,Östermalm,E-bike,1 helmet,f,t,f,Completed,2026-08-14,2026-08-21,2941.0,2.0


In [13]:
rentals['price'].describe()

count     352.000000
mean      847.348011
std       633.435513
min       154.500000
25%       406.000000
50%       668.250000
75%      1194.625000
max      3401.000000
Name: price, dtype: float64

In [14]:
print(f"{((rentals['price'].isna().sum() / len(rentals)) * 100).round(2)} % av uthyrningarna saknar pris")

12.66 % av uthyrningarna saknar pris


In [15]:
print(rentals['status'].unique())

<StringArray>
['Completed', 'No-show', 'Cancelled']
Length: 3, dtype: str


In [16]:
price_missing = rentals['price'].isna()
cancelled = rentals['status'] == 'Cancelled'
print(pd.crosstab(price_missing, cancelled))

status  False  True 
price               
False     352      0
True        4     47


In [17]:
tf_columns = [c for c in rentals.columns if set(rentals[c].dropna().unique()) <= {'t', 'f'}]

for c in tf_columns:
    rentals[c] = rentals[c].map({'t': True, 'f': False})

print(rentals[tf_columns].isna().sum())

is_member      0
insured        3
paid_online    0
dtype: int64


In [18]:
print(set(rentals['is_member'].dropna().unique()) <= {True, False})

True


In [19]:
print(rentals[['start_date', 'end_date']].isna().sum())

rentals['start_date'] = pd.to_datetime(rentals['start_date'], errors='coerce')
rentals['end_date'] = pd.to_datetime(rentals['end_date'], errors='coerce')

print(rentals[['start_date', 'end_date']].isna().sum())

start_date    0
end_date      0
dtype: int64
start_date    2
end_date      0
dtype: int64


In [20]:
print(rentals[['start_date', 'end_date']].dtypes)

start_date    datetime64[us]
end_date      datetime64[us]
dtype: object


In [21]:
rentals['rental_days'] = (rentals['end_date'] - rentals['start_date']).dt.days
print(rentals['rental_days'].describe())

count    401.000000
mean       2.955112
std        1.835206
min       -2.000000
25%        1.000000
50%        3.000000
75%        4.000000
max        7.000000
Name: rental_days, dtype: float64


In [22]:
print(rentals['rating'].isna().sum())

115


In [23]:
print(pd.crosstab(rentals['rating'].isna(), rentals['status']))

status  Cancelled  Completed  No-show
rating                               
False           0        288        0
True           47         46       22


In [24]:
rentals['has_rating'] = rentals['rating'].notna()
print(rentals['has_rating'].value_counts())

has_rating
True     288
False    115
Name: count, dtype: int64


In [25]:
print(rentals['rating'].describe())

count    288.000000
mean       4.225694
std        1.019547
min        1.000000
25%        4.000000
50%        5.000000
75%        5.000000
max        5.000000
Name: rating, dtype: float64


In [26]:
print(rentals['helmets_text'].value_counts(dropna=False))

helmets_text
1 helmet          146
2 helmets         127
No helmets         59
1 child helmet     47
3 helmets          20
NaN                 4
Name: count, dtype: int64
